# Capstone — Ranking Signal Analysis

**Question:** Which safe, content/search-level signals are associated with a page's visibility, clicks, or engagement in the FlyRank ML Internship warehouse?

**Decision this supports:** which signal(s) an editorial/SEO team should prioritize checking or fixing first when a page underperforms.

Run this notebook top to bottom with `HF_TOKEN` set in your environment (a Hugging Face **read** token with access to the gated warehouse dataset). It writes every table and chart the deployed paper needs into `outputs/`.

In [ ]:
import os
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

OUT = Path("outputs")
OUT.mkdir(exist_ok=True)

assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in your environment before running (gated dataset)."

con = duckdb.connect()
con.execute("SET hf_token = '{}'".format(os.environ["HF_TOKEN"]))
# DuckDB's hf:// filesystem support (httpfs) picks up HF_TOKEN from the environment too;
# the SET above covers duckdb versions that expose it as a config var instead.


## 1. Data — load & scope

Fill in the actual table path(s) from the dataset card / starter notebook 03 below.
Keep the date window and any exclusions explicit and public-safe (no client/domain names
in anything you print or save from this point forward — aggregate before you inspect).

In [ ]:
# TODO: replace with the real hf:// path(s) from the dataset card.
PAGES_TABLE = "hf://datasets/FlyRank/ml-internship-warehouse/pages/*.parquet"
METRICS_TABLE = "hf://datasets/FlyRank/ml-internship-warehouse/metrics/*.parquet"

# TODO: set your date window explicitly and document why (e.g. avoids a known
# tracking gap, matches a stable crawl period, etc.)
DATE_START = "2025-01-01"
DATE_END   = "2025-06-30"

raw = con.execute(f"""
    SELECT p.*, m.clicks, m.impressions, m.avg_position, m.date
    FROM read_parquet('{PAGES_TABLE}') p
    JOIN read_parquet('{METRICS_TABLE}') m USING (page_id)
    WHERE m.date BETWEEN '{DATE_START}' AND '{DATE_END}'
""").df()

print(raw.shape)
raw.head()

## 2. Exclusions (public-safe, documented)

State every filter and *why*. This section's row-count deltas go straight into
the paper's Data section.

In [ ]:
exclusion_log = []
df = raw.copy()
n0 = len(df)

# Example exclusions — adjust to what the real schema actually needs:
df = df[df["impressions"] > 0]
exclusion_log.append(("zero-impression rows (no visibility signal to learn from)", n0 - len(df)))

n1 = len(df)
df = df.dropna(subset=["avg_position", "clicks"])
exclusion_log.append(("missing position/clicks", n1 - len(df)))

pd.DataFrame(exclusion_log, columns=["exclusion", "rows_removed"]).to_csv(OUT / "exclusions.csv", index=False)
print(f"Kept {len(df)} of {n0} rows")

## 3. Label definition

Define the outcome explicitly. Ranking Signal Analysis works best as a regression
or ranked-correlation target (e.g. CTR relative to position-expected CTR, or
engagement rate), not a binary label — keep the target continuous so the
"associated with" framing in the paper stays honest.

In [ ]:
# Label: CTR gap vs. a position-expected CTR curve (standard SEO framing).
# Positive = over-performing its position, negative = under-performing.
df["ctr"] = df["clicks"] / df["impressions"]

expected_ctr_by_position = df.groupby(df["avg_position"].round())["ctr"].transform("median")
df["ctr_gap"] = df["ctr"] - expected_ctr_by_position

LABEL = "ctr_gap"
df[LABEL].describe()

## 4. Features

Only safe, structural/content signals — nothing that identifies a client, domain,
or private query. Adjust column names to the real schema.

In [ ]:
FEATURES = [
    "title_length",
    "meta_description_length",
    "word_count",
    "heading_count",
    "internal_link_count",
    "has_schema_markup",
    "content_age_days",
]
FEATURES = [c for c in FEATURES if c in df.columns]
print("Using features:", FEATURES)

model_df = df.dropna(subset=FEATURES + [LABEL]).copy()
len(model_df)

## 5. Validation design — grouped / time-aware split + leakage checks

Split by **page group / site**, not by row, so the same page's rows never appear
in both train and test — and prefer a time-aware split (train on the earlier part
of the window, test on the later part) so the model is validated the way it would
actually be used.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

model_df = model_df.sort_values("date")
cutoff = model_df["date"].quantile(0.8)

train_time = model_df[model_df["date"] <= cutoff]
test_time  = model_df[model_df["date"]  > cutoff]

# Leakage check: no page_id should appear in both halves.
overlap = set(train_time["page_id"]) & set(test_time["page_id"])
print(f"Time split — overlapping page_ids: {len(overlap)} (should be handled below if > 0)")

if overlap:
    # drop overlapping pages from train to keep the split clean
    train_time = train_time[~train_time["page_id"].isin(overlap)]

X_train, y_train = train_time[FEATURES], train_time[LABEL]
X_test,  y_test  = test_time[FEATURES],  test_time[LABEL]
print(len(X_train), len(X_test))

## 6. Baseline vs. model

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

baseline = DummyRegressor(strategy="median").fit(X_train, y_train)
model = GradientBoostingRegressor(random_state=42).fit(X_train, y_train)

results = {}
for name, m in [("baseline", baseline), ("model", model)]:
    pred = m.predict(X_test)
    results[name] = {
        "MAE": mean_absolute_error(y_test, pred),
        "R2": r2_score(y_test, pred),
    }

results_df = pd.DataFrame(results).T
results_df.to_csv(OUT / "model_vs_baseline.csv")
results_df

## 7. Signal report — feature importance / association strength

In [ ]:
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
importances.to_csv(OUT / "signal_importances.csv")

fig, ax = plt.subplots(figsize=(7, 4))
importances.plot(kind="barh", ax=ax)
ax.invert_yaxis()
ax.set_xlabel("Relative importance")
ax.set_title("Signal association with CTR gap")
fig.tight_layout()
fig.savefig(OUT / "signal_importances.png", dpi=150)

fig2, ax2 = plt.subplots(figsize=(6, 4))
ax2.bar(results_df.index, results_df["MAE"])
ax2.set_ylabel("MAE (lower is better)")
ax2.set_title("Model vs. baseline — held-out time split")
fig2.tight_layout()
fig2.savefig(OUT / "model_vs_baseline.png", dpi=150)

importances

## 8. Ranked recommendations (action playbook)

Write these once you've looked at `signal_importances.csv` for real — rank the
top 3-5 signals by importance, and for each state the *directional* recommendation
using decision-support language ("pages with X below N are associated with lower
CTR-gap and are worth review" — never "X causes higher ranking").

In [ ]:
recommendations = [
    # (signal, finding, action) — fill in from importances.head() + your own inspection
]
pd.DataFrame(recommendations, columns=["signal", "finding", "action"]).to_csv(OUT / "recommendations.csv", index=False)

## 9. Export everything the paper needs

Everything is now in `outputs/`:
- `exclusions.csv` -> Data section
- `model_vs_baseline.csv`, `model_vs_baseline.png` -> Results section
- `signal_importances.csv`, `signal_importances.png` -> Results + Ranked recommendations
- `recommendations.csv` -> Ranked recommendations

Open `../paper/index.html`, search for `[[FILL:`, and replace each placeholder with
the matching number/image from this folder. Then deploy the page and put the live
URL in `../submission/paper_url.txt`.